In [2]:
import sys
import os

# Add project root to Python path
sys.path.append(os.path.abspath(".."))

### 1. Import Libraries and Project Modules

This section imports all required libraries and project modules.

Custom modules from the `src` directory are used to ensure the workflow is modular and reusable.

The configuration file is also loaded to maintain consistency in parameters such as test size and random state.

In [3]:
import time
import pandas as pd

from src.data_loader import load_data
from src.preprocessing import build_preprocessor
from src.models import make_lr, make_rf
from src.evaluate import evaluate
from src.config import RANDOM_STATE, TEST_SIZE

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

### 2. Load Dataset and Perform Train-Test Split

The dataset is loaded using the custom data loader.

The features (`X`) and target variable (`y`) are separated.

A stratified train-test split is applied to preserve the original class imbalance.

This is important because fraud detection datasets are highly imbalanced, and random splitting could distort class proportions.

In [7]:
df = load_data("../data/creditcard.csv")

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain class counts:")
print(y_train.value_counts())

print("\nTest class counts:")
print(y_test.value_counts())

# Reduce training size for faster experimentation
sample_size = 50000

X_train_small = X_train.sample(n=sample_size, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

print("Reduced training size:", X_train_small.shape)

Train shape: (242085, 30)
Test shape: (42722, 30)

Train class counts:
Class
0    241667
1       418
Name: count, dtype: int64

Test class counts:
Class
0    42648
1       74
Name: count, dtype: int64
Reduced training size: (50000, 30)


### Interpretation

The dataset was successfully divided into training and testing subsets using stratified sampling.

The original class imbalance was preserved in both subsets, with fraudulent transactions remaining a very small minority.

This is important because model training and evaluation should reflect the true distribution of fraud cases in the dataset.

### 3. Train and Evaluate Logistic Regression Baseline

A Logistic Regression model is first used as the baseline supervised classifier.

This model is suitable as a starting point because it is simple, interpretable, and widely used in fraud detection studies.

The preprocessing pipeline is applied before training, and model performance is evaluated on the unseen test set.

In [8]:
pre = build_preprocessor()

lr_pipe = Pipeline([
    ("pre", pre),
    ("clf", make_lr())
])

print("Training Logistic Regression...")
start = time.time()

lr_pipe.fit(X_train, y_train)

end = time.time()
print(f"Logistic Regression training finished in {end - start:.2f} seconds")

y_pred_lr = lr_pipe.predict(X_test)
y_proba_lr = lr_pipe.predict_proba(X_test)[:, 1]

print("\n=== Logistic Regression ===")
evaluate(y_test, y_pred_lr, y_proba_lr)

Training Logistic Regression...
Logistic Regression training finished in 4.22 seconds

=== Logistic Regression ===
ROC-AUC : 0.9902794782683639
PR  AUC: 0.6993053415689807
              precision    recall  f1-score   support

           0     0.9999    0.9773    0.9885     42648
           1     0.0656    0.9189    0.1225        74

    accuracy                         0.9772     42722
   macro avg     0.5327    0.9481    0.5555     42722
weighted avg     0.9982    0.9772    0.9870     42722



### Interpretation

The Logistic Regression model achieved a high ROC-AUC and PR-AUC, indicating strong overall discrimination ability.

The recall for fraudulent transactions is very high, meaning that most fraud cases are successfully detected.

However, precision is very low, which indicates a high number of false positives.

This suggests that while the model is effective at identifying fraud, it may not be practical in real-world applications where excessive false alerts are costly.

### 4. Train and Evaluate Random Forest Baseline

A Random Forest classifier is used as a more advanced model capable of capturing non-linear relationships in the data.

This model is expected to improve precision compared to Logistic Regression while maintaining strong detection performance.

In [9]:
rf_pipe = Pipeline([
    ("pre", pre),
    ("clf", make_rf())
])

print("Training Random Forest on reduced sample...")
start = time.time()

rf_pipe.fit(X_train_small, y_train_small)

end = time.time()
print(f"Random Forest training finished in {end - start:.2f} seconds")

y_pred_rf = rf_pipe.predict(X_test)
y_proba_rf = rf_pipe.predict_proba(X_test)[:, 1]

print("\n=== Random Forest ===")
evaluate(y_test, y_pred_rf, y_proba_rf)

Training Random Forest on reduced sample...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   14.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   24.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s


Random Forest training finished in 25.07 seconds


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s



=== Random Forest ===
ROC-AUC : 0.9487224457152706
PR  AUC: 0.7456970206726284
              precision    recall  f1-score   support

           0     0.9996    0.9998    0.9997     42648
           1     0.8615    0.7568    0.8058        74

    accuracy                         0.9994     42722
   macro avg     0.9306    0.8783    0.9027     42722
weighted avg     0.9993    0.9994    0.9993     42722



[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


### Interpretation

The Random Forest model achieved much higher precision and F1-score for the fraud class than Logistic Regression.

This indicates that Random Forest produces far fewer false positives while still detecting a substantial proportion of fraud cases.

Although its recall is lower than Logistic Regression, the overall balance between precision and recall is much stronger, making it more suitable as a practical fraud detection baseline.

For computational efficiency, the Random Forest model was trained on a reduced sample of the training data, while evaluation was still performed on the full test set.

### Baseline Comparison Summary

The two baseline models show different strengths.

Logistic Regression favours fraud detection sensitivity, achieving very high recall but poor precision.

Random Forest provides a better balance, with strong precision, recall, and F1-score for the fraud class.

This comparison suggests that model choice in fraud detection depends on whether the priority is minimising missed fraud cases or reducing false alerts.
